# Lab 27-03 · A Smartwatch's Energy Ledger

Companion to [27-03 · Wearables I: The Smartwatch](../../chapters/27-03-wearables-smartwatch.md). Each part asks you to **predict** first, then run the cell. All parameters are the chapter's assumptions.

In [1]:
import math
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import watchmodel as wm

EB = wm.battery_energy_mwh(320, 3.85)
print(f'battery energy {EB:.0f} mWh')

battery energy 1232 mWh


## Part 1 · The floor and battery life

Example 27-03.1's day at the rails, with the floor as a parameter. *Predict* the life with a 4 mW floor before running.

In [2]:
def day_items(floor=6.0, aod_on=True):
    t_aod = 18 - 1 - 300*4/3600 - 90*6/3600 - 0.25
    items = [('floor', floor, 24), ('aod', 7.0 if aod_on else 0.0, t_aod),
             ('checks', 250, 300*4/3600), ('notifications', 300, 90*6/3600),
             ('app', 500, 0.25), ('workout', 150, 1.0), ('sleep tracking', 3, 6), ('heart rate', 2, 18)]
    return items

for fl in (6.0, 4.0):
    e = wm.daily_energy_mwh(day_items(fl))
    print(f'floor {fl:.0f} mW: E_day = {e:.1f} mWh, life = {wm.battery_life_h(EB, e, 0.85):.1f} h')

floors = np.linspace(2, 10, 81)
lives = [wm.battery_life_h(EB, wm.daily_energy_mwh(day_items(f)), 0.85) for f in floors]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(floors, lives)
ax.set_xlabel('floor (mW)'); ax.set_ylabel('battery life (h)'); ax.grid(True, alpha=0.3)
fig.tight_layout()

floor 6 mW: E_day = 715.2 mWh, life = 35.1 h
floor 4 mW: E_day = 667.2 mWh, life = 37.7 h


## Part 2 · Where the BLE current stops falling

*Predict* the interval at which the average current is twice its interval-independent part, $I_{sl} + I_{rx}s$ = 1.86 µA.

In [3]:
t_e = wm.ble_airtime_us(0, 1)
q = wm.ble_event_charge_uc(1.2, 250, 3.6, t_e, 150, 4.1, t_e)
floor_ua = 1.5 + 1e3*3.6*100e-6
T2 = q / floor_ua
print(f'Q_ev = {q:.3f} uC; floor = {floor_ua:.2f} uA; twice the floor at T = {T2:.3f} s')

T = np.logspace(-2, 1.3, 200)
I = [wm.ble_avg_current_ua(q, t, 1.5, 3.6, 100) for t in T]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.loglog(T, I, label='average current')
ax.axhline(floor_ua, ls='--', color='gray', label='sleep + window widening')
ax.set_xlabel('connection interval (s)'); ax.set_ylabel('current (uA)'); ax.legend(); ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()

Q_ev = 1.456 uC; floor = 1.86 uA; twice the floor at T = 0.783 s


## Part 3 · Refresh and the always-on display

*Predict* the always-on display power at 10 Hz (Example 27-03.2's panel).

In [4]:
A, EFF, P0, EF = 1.4e-3, 10.0, 2.0, 0.3
for f in (1, 10, 60):
    print(f'{f:>2} Hz: {wm.display_power_mw(50, A, 0.15, EFF, P0, EF, f):.2f} mW')

fr = np.linspace(1, 60, 60)
fig, ax = plt.subplots(figsize=(6, 3.5))
for apl in (0.15, 1.0):
    ax.plot(fr, [wm.display_power_mw(50, A, apl, EFF, P0, EF, f) for f in fr], label=f'APL {apl}')
ax.set_xlabel('refresh rate (Hz)'); ax.set_ylabel('display power (mW)'); ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()

 1 Hz: 5.60 mW
10 Hz: 8.30 mW
60 Hz: 23.30 mW


## Part 4 · Batching against the hub

*Predict* the smallest batch at which an AP spending 3 µJ per sample matches the 0.208 mW hub.

In [5]:
FS, EW = 50, 0.5
p_hub = wm.hub_power_mw(0.15, FS, 0.001, 1/60, EW)
N = wm.batch_for_power(FS, 0.003, EW, p_hub)
print(f'hub {p_hub:.3f} mW; batch {N} samples = {N/FS:.2f} s of latency')

n = np.arange(1, 1001)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.loglog(n, wm.ap_batched_power_mw(FS, 0.003, EW, n), label='AP, 3 uJ/sample')
ax.loglog(n, wm.ap_batched_power_mw(FS, 0.020, EW, n), label='AP, 20 uJ/sample')
ax.axhline(p_hub, ls='--', color='gray', label='always-on hub')
ax.axvline(7, ls=':', color='k', label='150 ms latency limit')
ax.set_xlabel('batch size N (samples)'); ax.set_ylabel('power (mW)'); ax.legend(); ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()

hub 0.208 mW; batch 429 samples = 8.58 s of latency


## Discussion

Compare your predictions with the expected results in the [lab README](README.md). Which of the four knobs would you spend engineering effort on first for this watch, and why?